### Import required modules

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

import numpy as np

sns.set_style("whitegrid")

## Load all dataset using python

In [ ]:
train_df = pd.read_csv("./datasets/train.csv")
val_df = pd.read_csv("./datasets/val.csv")
test_df = pd.read_csv("./datasets/test.csv")

train_df.head()

## Use only selected features for linear regression

In [ ]:
selected_columns = [
    "Year",
    "dengue_total",
    "Location",
    "Month",
    "monthly_avg_temperature",
    "avg_daily_rain",
    "avg_daily_humidity",
    "avg_daily_soil_moisture",
    "avg_daily_soil_temperature",
    "avg_daily_snowfall",
    "avg_daily_precipitation"
]

train = train_df[selected_columns]
val = val_df[selected_columns]
test = test_df[selected_columns]

all_locations = sorted(pd.concat([train["Location"], val["Location"], test["Location"]]).unique())
all_months = sorted(pd.concat([train["Month"], val["Month"], test["Month"]]).unique())

train_loc = pd.get_dummies(train["Location"].astype(pd.CategoricalDtype(all_locations)), dtype=int)
train_month = pd.get_dummies(train["Month"].astype(pd.CategoricalDtype(all_months)), dtype=int)

val_loc = pd.get_dummies(val["Location"].astype(pd.CategoricalDtype(all_locations)), dtype=int)
val_month = pd.get_dummies(val["Month"].astype(pd.CategoricalDtype(all_months)), dtype=int)

test_loc = pd.get_dummies(test["Location"].astype(pd.CategoricalDtype(all_locations)), dtype=int)
test_month = pd.get_dummies(test["Month"].astype(pd.CategoricalDtype(all_months)), dtype=int)

train = pd.concat([train.drop(columns=["Month", "Location"]), train_loc, train_month], axis=1)
val = pd.concat([val.drop(columns=["Month", "Location"]), val_loc, val_month], axis=1)
test = pd.concat([test.drop(columns=["Month", "Location"]), test_loc, test_month], axis=1)

train.head()

# Extract X and Y from data

### Normalize and prepare


In [ ]:
def normalize_train(val):
    mean = np.nanmean(val, axis=0)
    std = np.nanstd(val, axis=0)
    return (val - mean / std), mean, std

def normalize_pred(val, mean, std):
    return (val - mean)/std

Training set

In [ ]:
x_train= train.drop(columns=["dengue_total"])
# all the features
features = x_train.columns.to_list()

x_train, x_mean, x_std = normalize(x_train.to_numpy())
y_train, y_mean, y_std = normalize(train["dengue_total"].to_numpy())

print("Columns of features:")
print(features)

print(f"Shape of x: {x_train.shape}")

Validation and Test set

In [ ]:
# Validation Set
x_val = normalize(val.drop(columns=["dengue_total"]).to_numpy())
y_val = normalize(val["dengue_total"].to_numpy())

# Testing Set
x_test = normalize(test.drop(columns=["dengue_total"]).to_numpy())
y_test = test["dengue_total"].to_numpy()

print(f"Validation and test sets are ready to use!")

# LinearRegression Architecture

In [ ]:
import numpy as np

class LinearRegression:
    def __init__(self, shape, lr=0.01):
        self.w = np.random.randn(shape) * 0.01
        self.b = 0
        self.lr = lr

    def fit(self, x, y):
        loss = []

        for x_r, y_r in zip(x, y):
            y_pred = np.dot(x_r, self.w) + self.b

            error = y_pred - y_r

            # gradient update
            self.w -= self.lr * error * x_r
            self.b -= self.lr * error

            mse = error ** 2
            loss.append(mse)

        return np.mean(loss)

    def predict(self, x):
        return np.dot(x, self.w) + self.b

# Training Linear Model in 100 Epochs

In [ ]:
print(x_train.min(), x_train.max())
print(x_train.mean(), x_train.std())

In [ ]:
model = LinearRegression(x_train.shape[1], 0.01)

train_losses = []
val_losses = []

# 100 epochs
for epoch in range(1, 50):

    # training on train set
    train_loss = model.fit(x=x_train, y=y_train)
    train_losses.append(train_loss)

    # validation
    preds = model.predict(x_val)
    val_loss = np.mean((preds - y_val) ** 2)
    val_losses.append(val_loss)
    if epoch%10 == 0:
        print(f"Epoch {epoch}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}")

In [ ]:
train_losses

In [ ]:
val_losses